In [1]:
from datetime import date

import nflreadpy as nfl
import pandas as pd

current_year = date.today().year
seasons = list(range(current_year - 4, current_year + 1))
print("Requested seasons:", seasons)

Requested seasons: [2022, 2023, 2024, 2025, 2026]


In [2]:
from nfl_analyzer.fantasy import (
    season_dst_fantasy_points,
    season_fantasy_points,
    yoy_trend_table,
)
from nfl_analyzer.filters import (
    aggregate_by_player_id,
    exclude_incoming_rookies,
    exclude_significant_injury,
    filter_draftable_positions,
    skill_depth_ids,
    starting_qb_ids,
)


def load_stats(load_fn, seasons: list[int]) -> pd.DataFrame:
    """Load stats for as many of the requested seasons as are available."""
    frames: list[pd.DataFrame] = []
    loaded: list[int] = []
    for season in seasons:
        try:
            frame = load_fn(season).to_pandas()
        except Exception as exc:  # noqa: BLE001 - missing seasons are expected
            print(f"Skipping {season}: {exc}")
            continue
        if frame.empty:
            print(f"Skipping {season}: no rows returned")
            continue
        frames.append(frame)
        loaded.append(season)
    if not frames:
        raise RuntimeError("No seasons returned data")
    print("Loaded seasons:", loaded)
    return pd.concat(frames, ignore_index=True)


team_stats = load_stats(nfl.load_team_stats, seasons)
player_stats = load_stats(nfl.load_player_stats, seasons)
print("team_stats shape:", team_stats.shape)
print("player_stats shape (before inactive filter):", player_stats.shape)

players = nfl.load_players().to_pandas()
latest_loaded_season = int(player_stats["season"].max())
eligible_players = players[
    (players["status"] != "RET")
    & (players["last_season"].fillna(0) >= latest_loaded_season)
][["gsis_id"]].dropna()

player_stats = player_stats.merge(
    eligible_players,
    left_on="player_id",
    right_on="gsis_id",
    how="inner",
).drop(columns=["gsis_id"])
print(
    "player_stats shape (after excluding RET / stale last_season):",
    player_stats.shape,
)
print("Eligibility cutoff last_season >=", latest_loaded_season)

before = len(player_stats)
player_stats = exclude_incoming_rookies(player_stats, players, latest_loaded_season)
print(
    f"After excluding incoming rookies (draft_year > {latest_loaded_season}):",
    player_stats.shape,
    f"(removed {before - len(player_stats)} rows)",
)

injuries = nfl.load_injuries(latest_loaded_season).to_pandas()
before = len(player_stats)
player_stats = exclude_significant_injury(
    player_stats, injuries, latest_loaded_season
)
print(
    f"After excluding significant injury (>2 REG Out weeks in {latest_loaded_season}):",
    player_stats.shape,
    f"(removed {before - len(player_stats)} rows)",
)

before = len(player_stats)
player_stats = filter_draftable_positions(player_stats)
print(
    "After keeping draftable positions only:",
    player_stats.shape,
    f"(removed {before - len(player_stats)} rows)",
)

player_stats_weekly = player_stats.copy()
before = len(player_stats)
player_stats = aggregate_by_player_id(player_stats)
print(
    "After grouping by player_id:",
    player_stats.shape,
    f"(from {before} weekly rows)",
)
print("Remaining positions:\n", player_stats["position"].value_counts())

season_fp = season_fantasy_points(player_stats_weekly)
player_yoy = yoy_trend_table(season_fp)
print("YoY trend rows:", len(player_yoy))

dst_season_fp = season_dst_fantasy_points(team_stats)
dst_yoy = yoy_trend_table(dst_season_fp, id_col="team")
print("D/ST YoY trend rows:", len(dst_yoy))

depth_charts = nfl.load_depth_charts(latest_loaded_season).to_pandas()
starter_qbs = starting_qb_ids(depth_charts)
skill_depth = skill_depth_ids(depth_charts)
print(f"Depth-chart QB starters: {len(starter_qbs)}")
print(f"Depth-chart skill players (rank <= 2): {len(skill_depth)}")


Skipping 2026: Failed to download https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet: 404 Client Error: Not Found for url: https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet
Loaded seasons: [2022, 2023, 2024, 2025]
Skipping 2026: Failed to download https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_week_2026.parquet: 404 Client Error: Not Found for url: https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_week_2026.parquet
Loaded seasons: [2022, 2023, 2024, 2025]
team_stats shape: (2278, 138)
player_stats shape (before inactive filter): (75879, 150)
player_stats shape (after excluding RET / stale last_season): (66432, 150)
Eligibility cutoff last_season >= 2025
After excluding incoming rookies (draft_year > 2025): (66432, 150) (removed 0 rows)
After excluding significant injury (>2 REG Out weeks in 2025): (60935, 150) (rem

In [3]:
team_stats

,season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,...,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards
0,2022,1,ARI,REG,2022_01_KC_ARI,KC,24,38,205,2,...,255,0,0,0,0,0,5,44,0,211
1,2022,1,ATL,REG,2022_01_NO_ATL,NO,20,33,215,0,...,180,2,0,0,0,3,1,12,0,168
2,2022,1,BAL,REG,2022_01_BAL_NYJ,NYJ,17,30,213,3,...,291,3,0,0,1,3,2,30,0,241
3,2022,1,BUF,REG,2022_01_BUF_LA,LA,26,31,297,3,...,0,0,0,0,0,0,0,0,0,0
4,2022,1,CAR,REG,2022_01_CLE_CAR,CLE,16,27,235,1,...,250,0,0,0,0,0,5,29,0,221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,2025,21,LA,POST,2025_21_LA_SEA,SEA,22,35,374,3,...,84,1,0,0,0,2,0,0,0,84
2274,2025,21,NE,POST,2025_21_NE_DEN,DEN,10,21,86,0,...,303,1,3,1,0,1,3,8,0,295
2275,2025,21,SEA,POST,2025_21_LA_SEA,LA,25,36,346,3,...,226,5,0,0,0,3,2,0,0,226
2276,2025,22,NE,POST,2025_22_SEA_NE,SEA,27,43,295,2,...,356,4,0,0,0,6,2,16,0,340


In [4]:
player_stats

,player_id,player_name,player_display_name,position,position_group,headshot_url,team,games,season_min,season_max,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0022942,P.Rivers,Philip Rivers,QB,QB,https://static.www.nfl.com/image/private/f_aut...,IND,3,2025,2025,...,0,0,0,0,0,0,0,0,31.66,31.66
1,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,PIT,52,2022,2025,...,0,0,0,0,0,0,0,0,723.70,724.70
2,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,BUF,51,2022,2025,...,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0024243,M.Lewis,Marcedes Lewis,TE,TE,https://static.www.nfl.com/image/private/f_aut...,DEN,18,2022,2025,...,0,0,0,0,0,0,0,0,29.70,40.70
4,00-0025565,N.Folk,Nick Folk,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,NYJ,64,2022,2025,...,0,0,0,1,1,3,0,78,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,00-0040737,T.Ferguson,Terrance Ferguson,TE,TE,https://static.www.nfl.com/image/upload/f_auto...,LA,13,2025,2025,...,0,0,0,0,0,0,0,0,43.00,55.00
715,00-0040739,E.Arroyo,Elijah Arroyo,TE,TE,https://static.www.nfl.com/image/upload/f_auto...,SEA,13,2025,2025,...,0,0,0,0,0,0,0,0,21.90,36.90
716,00-0040743,T.Shough,Tyler Shough,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NO,11,2025,2025,...,0,0,0,0,0,0,0,0,157.96,157.96
717,00-0040782,I.Bond,Isaiah Bond,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,CLE,15,2025,2025,...,0,0,0,0,0,0,0,0,36.70,54.70


In [5]:
qb_rankings = player_stats.loc[player_stats["position"] == "QB"].copy()
before = len(qb_rankings)
qb_rankings = qb_rankings[
    qb_rankings["player_id"].astype(str).isin(starter_qbs)
].copy()
print(f"QB starters after depth-chart filter: {len(qb_rankings)} (from {before})")
qb_rankings["total_tds"] = (
    qb_rankings["passing_tds"].fillna(0) + qb_rankings["rushing_tds"].fillna(0)
)
qb_rankings = qb_rankings.merge(player_yoy, on="player_id", how="left")
qb_rankings = (
    qb_rankings.sort_values("passing_yards", ascending=False)[
        [
            "player_id",
            "player_display_name",
            "team",
            "position",
            "games",
            "passing_yards",
            "passing_tds",
            "rushing_tds",
            "total_tds",
            "fantasy_points_prev",
            "fantasy_points_latest",
            "yoy_pct",
            "trend",
        ]
    ]
    .reset_index(drop=True)
)
qb_rankings


QB starters after depth-chart filter: 27 (from 92)


,player_id,player_display_name,team,position,games,passing_yards,passing_tds,rushing_tds,total_tds,fantasy_points_prev,fantasy_points_latest,yoy_pct,trend
0,00-0033873,Patrick Mahomes,KC,QB,73,19381,134,13,147,333.0,319.5,-0.040541,↓
1,00-0033106,Jared Goff,DET,QB,72,19356,135,2,137,347.0,369.0,0.063401,↑
2,00-0034857,Josh Allen,BUF,QB,74,18185,132,56,188,443.0,406.0,-0.083521,↓
3,00-0026498,Matthew Stafford,LA,QB,63,16357,112,1,113,288.0,442.0,0.534722,↑
4,00-0036355,Justin Herbert,LAC,QB,66,16144,96,7,103,270.5,338.0,0.249538,↑
5,00-0030565,Geno Smith,LV,QB,65,15504,92,4,96,262.5,200.0,-0.238095,↓
6,00-0036389,Jalen Hurts,PHI,QB,72,15409,98,60,158,400.0,345.0,-0.137500,↓
7,00-0034855,Baker Mayfield,TB,QB,66,15271,113,6,119,318.0,310.0,-0.025157,↓
8,00-0036971,Trevor Lawrence,JAX,QB,63,14893,94,21,115,296.0,380.0,0.283784,↑
9,00-0033077,Dak Prescott,DAL,QB,57,14820,108,7,115,411.0,369.0,-0.102190,↓


In [6]:
kicker_rankings = player_stats.loc[player_stats["position"] == "K"].copy()
kicker_rankings = kicker_rankings.merge(player_yoy, on="player_id", how="left")
kicker_rankings = (
    kicker_rankings.sort_values("fg_made", ascending=False)[
        [
            "player_id",
            "player_display_name",
            "team",
            "position",
            "games",
            "fg_made",
            "fg_att",
            "fantasy_points_prev",
            "fantasy_points_latest",
            "yoy_pct",
            "trend",
        ]
    ]
    .reset_index(drop=True)
)
kicker_rankings


,player_id,player_display_name,team,position,games,fg_made,fg_att,fantasy_points_prev,fantasy_points_latest,yoy_pct,trend
0,00-0032726,Ka'imi Fairbairn,HOU,K,67,147,163,44.0,74.5,0.693182,↑
1,00-0031492,Jason Myers,SEA,K,72,145,166,67.5,76.5,0.133333,↑
2,00-0037224,Cameron Dicker,LAC,K,65,135,145,58.0,71.0,0.224138,↑
3,00-0035358,Chase McLaughlin,TB,K,70,127,144,50.5,48.5,-0.039604,↓
4,00-0033303,Harrison Butker,KC,K,70,126,144,63.5,59.5,-0.062992,↓
5,00-0031136,Chris Boswell,PIT,K,66,120,138,50.5,52.0,0.029703,↑
6,00-0032569,Wil Lutz,DEN,K,71,116,138,55.5,56.0,0.009009,→
7,00-0034161,Daniel Carlson,LV,K,68,116,134,57.0,37.5,-0.342105,↓
8,00-0033787,Jake Elliott,PHI,K,76,115,136,59.5,46.0,-0.226891,↓
9,00-0037692,Brandon Aubrey,DAL,K,52,113,128,63.5,65.5,0.031496,↑


In [7]:
skill_rankings = player_stats.loc[
    player_stats["position"].isin(["RB", "WR", "TE", "FB"])
].copy()
before = len(skill_rankings)
skill_rankings = skill_rankings[
    skill_rankings["player_id"].astype(str).isin(skill_depth)
].copy()
print(
    f"Skill players after depth-chart filter (rank <= 2): "
    f"{len(skill_rankings)} (from {before})"
)
skill_rankings["total_yards"] = (
    skill_rankings["rushing_yards"].fillna(0)
    + skill_rankings["receiving_yards"].fillna(0)
)
skill_rankings = skill_rankings.merge(player_yoy, on="player_id", how="left")
skill_rankings = (
    skill_rankings.sort_values("total_yards", ascending=False)[
        [
            "player_id",
            "player_display_name",
            "team",
            "position",
            "games",
            "rushing_yards",
            "receiving_yards",
            "total_yards",
            "fantasy_points_prev",
            "fantasy_points_latest",
            "yoy_pct",
            "trend",
        ]
    ]
    .reset_index(drop=True)
)
skill_rankings


Skill players after depth-chart filter (rank <= 2): 185 (from 582)


,player_id,player_display_name,team,position,games,rushing_yards,receiving_yards,total_yards,fantasy_points_prev,fantasy_points_latest,yoy_pct,trend
0,00-0034844,Saquon Barkley,PHI,RB,69,6138,1346,7484,191.5,208.5,0.088773,↑
1,00-0032764,Derrick Henry,BAL,RB,69,6491,955,7446,235.0,285.5,0.214894,↑
2,00-0033280,Christian McCaffrey,SF,RB,62,4591,2693,7284,363.5,363.0,-0.001376,→
3,00-0035700,Josh Jacobs,GB,RB,64,4852,1363,6215,156.5,208.0,0.329073,↑
4,00-0037248,James Cook,BUF,RB,75,4886,1293,6179,202.0,300.5,0.487624,↑
...,...,...,...,...,...,...,...,...,...,...,...,...
180,00-0040177,Jordan James,SF,RB,1,28,7,35,NaN,NaN,NaN,NaN
181,00-0039407,Dylan Laube,LV,RB,23,9,18,27,NaN,2.5,NaN,—
182,00-0039305,Jack Westover,NE,TE,5,0,8,8,NaN,0.5,NaN,—
183,00-0040649,Robbie Ouzts,SEA,RB,5,0,0,0,NaN,0.0,NaN,—


In [8]:
dst_rankings = (
    dst_season_fp.groupby("team", as_index=False)["fantasy_points"]
    .sum()
    .rename(columns={"fantasy_points": "fantasy_points_total"})
    .merge(dst_yoy, on="team", how="left")
    .sort_values("fantasy_points_total", ascending=False)
    .reset_index(drop=True)
)[
    [
        "team",
        "fantasy_points_total",
        "fantasy_points_prev",
        "fantasy_points_latest",
        "yoy_pct",
        "trend",
    ]
]
dst_rankings


,team,fantasy_points_total,fantasy_points_prev,fantasy_points_latest,yoy_pct,trend
0,PIT,417.0,109.0,104.0,-0.045872,↓
1,HOU,407.0,94.0,111.0,0.180851,↑
2,DEN,399.0,98.0,100.0,0.020408,↑
3,BUF,396.0,115.0,78.0,-0.321739,↓
4,MIN,396.0,89.0,95.0,0.067416,↑
5,DAL,392.0,105.0,66.0,-0.371429,↓
6,PHI,391.0,81.0,92.0,0.135802,↑
7,BAL,387.0,122.0,74.0,-0.393443,↓
8,SEA,374.0,87.0,103.0,0.183908,↑
9,IND,368.0,103.0,85.0,-0.174757,↓


In [9]:
from nfl_analyzer.draft import build_board, build_ideal_team, team_bye_weeks

bye_by_team = team_bye_weeks(nfl.load_schedules(latest_loaded_season).to_pandas())
draft_board = build_board(
    qb_rankings,
    skill_rankings,
    kicker_rankings,
    dst_rankings,
    bye_by_team,
)
ideal_team = build_ideal_team(draft_board)
print(ideal_team["position"].value_counts().to_dict())
ideal_team


{'RB': 3, 'WR': 3, 'TE': 3, 'QB': 2, 'K': 2, 'D/ST': 1}


,pick,player_display_name,position,team,bye_week,fantasy_points_latest,yoy_pct,trend
0,1,Josh Allen,QB,BUF,7,406.0,-0.083521,↓
1,2,Christian McCaffrey,RB,SF,14,363.0,-0.001376,→
2,3,Jonathan Taylor,RB,IND,11,341.0,1.444444,↑
3,4,Bijan Robinson,RB,ATL,5,337.5,0.577103,↑
4,5,Puka Nacua,WR,LA,8,320.5,0.249513,↑
5,6,Jaxon Smith-Njigba,WR,SEA,8,318.5,1.831111,↑
6,7,Ja'Marr Chase,WR,CIN,10,265.5,0.201357,↑
7,8,Kyle Pitts,TE,ATL,5,160.0,0.560976,↑
8,9,Hunter Henry,TE,NE,14,142.0,0.526882,↑
9,10,Matthew Stafford,QB,LA,8,442.0,0.534722,↑


In [10]:
from nfl_analyzer.draft import simulate_snake_draft
from nfl_analyzer.league import PICK_SLOT

draft_log, my_simulated_team = simulate_snake_draft(draft_board)
print(f"Snake draft: {len(draft_log)} picks across 12 teams")
print(f"My team (slot {PICK_SLOT}) position counts:")
print(my_simulated_team["position"].value_counts().to_dict())
print("My picks from the log:")
display(draft_log.loc[draft_log["draft_team"] == PICK_SLOT])
my_simulated_team


Snake draft: 89 picks across 12 teams
My team (slot 8) position counts:
{'RB': 3, 'TE': 3, 'QB': 1, 'WR': 1}
My picks from the log:


,overall_pick,round,draft_team,player_display_name,position,nfl_team,bye_week,fantasy_points_latest,yoy_pct,trend
7,8,1,8,Josh Allen,QB,BUF,7,406.0,-0.083521,↓
16,17,2,8,Ja'Marr Chase,WR,CIN,10,265.5,0.201357,↑
31,32,3,8,Jaylen Warren,RB,PIT,5,190.0,0.237785,↑
40,41,4,8,Kyle Pitts,TE,ATL,5,160.0,0.560976,↑
55,56,5,8,J.K. Dobbins,RB,DEN,12,108.5,9.850000,↑
64,65,6,8,Brenton Strange,TE,JAX,8,89.0,7.476190,↑
79,80,7,8,Austin Hooper,TE,NE,14,43.5,0.474576,↑
88,89,8,8,Hunter Luepke,RB,DAL,10,24.5,2.266667,↑


,pick,player_display_name,position,team,bye_week,fantasy_points_latest,yoy_pct,trend
0,1,Josh Allen,QB,BUF,7,406.0,-0.083521,↓
1,2,Ja'Marr Chase,WR,CIN,10,265.5,0.201357,↑
2,3,Jaylen Warren,RB,PIT,5,190.0,0.237785,↑
3,4,Kyle Pitts,TE,ATL,5,160.0,0.560976,↑
4,5,J.K. Dobbins,RB,DEN,12,108.5,9.850000,↑
5,6,Brenton Strange,TE,JAX,8,89.0,7.476190,↑
6,7,Austin Hooper,TE,NE,14,43.5,0.474576,↑
7,8,Hunter Luepke,RB,DAL,10,24.5,2.266667,↑
